In [ ]:
# --- repo bootstrap (Colab + local compatible) ---
import sys
from pathlib import Path

repo = Path.cwd()

if repo.name == "notebooks":
    repo = repo.parent

if not (repo / "src").exists():
    !git clone https://github.com/thinkthoughts/ion-transport-waveform-pipeline.git
    %cd ion-transport-waveform-pipeline
    repo = Path.cwd()

if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

print("Repo root:", repo)


# 01 — Well Positioning

Solve electrode voltages that place an approximate harmonic potential well at selected axial target positions.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from src.ion_transport_waveform.config import TrapConfig
from src.ion_transport_waveform.trap_model import (
    gaussian_electrode_basis,
    potential_from_voltages,
    electric_field,
    curvature_at_target,
)
from src.ion_transport_waveform.waveform_solver import solve_voltages_for_target_well

fig_dir = repo / "figures"
fig_dir.mkdir(exist_ok=True)


In [ ]:
cfg = TrapConfig()

x = np.linspace(-420e-6, 420e-6, 1400)
electrode_positions = np.arange(-5, 6) * cfg.electrode_pitch_m
basis = gaussian_electrode_basis(x, electrode_positions, cfg.basis_width_m)


In [ ]:
targets_um = np.array([-160, -80, 0, 80, 160], dtype=float)
targets = targets_um * 1e-6

solutions = []
for target_x in targets:
    v = solve_voltages_for_target_well(
        basis=basis,
        x_grid=x,
        target_x=target_x,
        voltage_limit=cfg.voltage_limit_v,
        ridge=1e-5,
    )
    phi = potential_from_voltages(basis, v)
    min_x = x[np.argmin(phi)]
    solutions.append({
        "target_x": target_x,
        "voltages": v,
        "potential": phi,
        "min_x": min_x,
    })


In [ ]:
plt.figure(figsize=(8,4))
for s in solutions:
    phi = s["potential"]
    phi = (phi - phi.min()) / (phi.max() - phi.min())
    plt.plot(x*1e6, phi, label=f"{s['target_x']*1e6:.0f} µm")
plt.xlabel("position (µm)")
plt.ylabel("normalized potential")
plt.legend()
plt.tight_layout()
plt.savefig(fig_dir / "well_positioning.png")
plt.show()
